In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def plot_cumulative_column_by_trace(df, column, trace_column, trace_ranges, labels, colors, title, save_path, xlim=None):
    """
    Generate and save cumulative probability plots as points filtered by trace number ranges.
    """
    plt.figure(figsize=(10, 6))

    for (start, end), label, color in zip(trace_ranges, labels, colors):
        data = df[(df[trace_column] >= start) & (df[trace_column] <= end)][column].dropna()
        if len(data) == 0:
            continue
        sorted_data = np.sort(data)
        cum_prob = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        plt.scatter(sorted_data, cum_prob, label=label, color=color, s=10)

    plt.xlabel(column)
    plt.ylabel("Cumulative Probability")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    if xlim:
        plt.xlim(xlim)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved: {save_path}")


def process_directory_for_trace_plots(directory_path, output_dir=None):
    """
    Processes all CSV files in a directory and generates cumulative plots based on trace ranges
    for:
    - 'peak amp (pA)'
    - Interevent interval (ms), derived from 'Inst. Freq. (Hz)'
    """
    if output_dir is None:
        output_dir = directory_path

    os.makedirs(output_dir, exist_ok=True)

    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))
    if not csv_files:
        print("No CSV files found in the directory.")
        return

    # Define trace ranges
    trace_ranges = [
        (1, 10),
        (12, 22),
        (24, 34)
    ]
    labels = [
        "Trace 1–10",
        "Trace 12–22",
        "Trace 24–34"
    ]
    colors = ["black", "red", "blue"]

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            base_name = os.path.splitext(os.path.basename(csv_file))[0]

            # Compute interevent interval (ms) from frequency (Hz)
            if "Inst. Freq. (Hz)" in df.columns:
                df["Interevent interval (ms)"] = 1000 / df["Inst. Freq. (Hz)"]
                df.replace([np.inf, -np.inf], np.nan, inplace=True)

            # Peak Amplitude Plot
            save_path_amp = os.path.join(output_dir, f"{base_name}_peak_amp.png")
            plot_cumulative_column_by_trace(
                df,
                column="peak amp (pA)",
                trace_column="trace",
                trace_ranges=trace_ranges,
                labels=labels,
                colors=colors,
                title=f"Cumulative Probability of Peak Amplitude\n{base_name}",
                save_path=save_path_amp,
                xlim=(-500, 0)
            )

            # Interevent Interval (from frequency) Plot
            save_path_iei = os.path.join(output_dir, f"{base_name}_interevent_interval_from_freq.png")
            plot_cumulative_column_by_trace(
                df,
                column="Interevent interval (ms)",
                trace_column="trace",
                trace_ranges=trace_ranges,
                labels=labels,
                colors=colors,
                title=f"Cumulative Probability of Interevent Interval (from Frequency)\n{base_name}",
                save_path=save_path_iei,
                xlim=(0, None)
            )

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

In [ ]:
# Example usage:
process_directory_for_trace_plots("./data/MCcsv")